<a href="https://colab.research.google.com/github/Bix4UMD/BIOI611_lab/blob/main/docs/FASTQ_PHRED.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## What is PHRED Scores

A Phred score is a measure of the probability that a base call in a DNA sequencing read is incorrect. It is a logarithmic scale, meaning that a small change in the Phred score represents a large change in the probability of an error.

$$Q = -10 \cdot \log_{10}(P)$$

Where:

-   Q is the PHRED score.

-   P is the probability that the base was called incorrectly.

For example:

-   **Q = 20**: This corresponds to a 1 in 100 probability of an incorrect base call, or an accuracy of 99%.

-   **Q = 30**: This corresponds to a 1 in 1000 probability of an incorrect base call, or an accuracy of 99.9%.

-   **Q = 40**: This corresponds to a 1 in 10,000 probability of an incorrect base call, or an accuracy of 99.99%.

```{r}
# Print the header
cat(sprintf("%-5s\t\t%-10s\n", "Phred", "Prob of"))
cat(sprintf("%-5s\t\t%-10s\n", "score", "Incorrect call"))

# Loop through Phred scores from 0 to 41
for (phred in 0:41) {
  cat(sprintf("%-5d\t\t%0.5f\n", phred, 10^(phred / -10)))
}
```

## What is ASCII

ASCII (American Standard Code for Information Interchange) is used to represent characters in computers. We can represent Phred scores using ASCII characters. The advantage is that the quality information can be esisly stored in text based FASTQ file.

Not all [ASCII characters](https://www.columbia.edu/kermit/ascii.html) are printable. The first printable ASCII character is `!` and the decimal code for the character for `!` is 33.


```{r}
# Store output in a vector to fit on a slide
output <- c(sprintf("%-8s  %-8s", "Character", "ASCII #"))

# Loop through ASCII values from 33 to 89
for (i in 33:89) {
  output <- c(output, sprintf("%-8s  %-8d", intToUtf8(i), i))
}

# Print the output in a single block (e.g., to fit on a slide)
cat(paste(output, collapse = "\n"))
```
## Phred scores in FASTQ file

In a FASTQ file, Phred scores are represented as ASCII characters. These characters are converted back to numeric values (PHRED scores) based on the encoding scheme used:

1.  **PHRED+33 Encoding (Sanger/Illumina 1.8+)**:

    -   The ASCII character for a quality score Q is calculated as:

        ASCII character=chr(Q+33)

    -   For example:

        -   A PHRED score of 30 is encoded as `chr(30 + 33) = chr(63)`, which corresponds to the ASCII character `?`.

2.  **PHRED+64 Encoding (Illumina 1.3-1.7)**:

    -   The ASCII character for a quality score QQQ is calculated as:
         
         ASCII character=chr(Q+64)

    -   For example:

        -   A PHRED score of 30 is encoded as `chr(30 + 64) = chr(94)`, which corresponds to the ASCII character `^`.


```{r}
# Print the header
cat(sprintf("%-5s\t\t%-10s\t%-6s\t\t%-10s\n", "Phred", "Prob. of", "ASCII", "ASCII"))
cat(sprintf("%-5s\t\t%-10s\t%-6s\t%-10s\n", "score", "Error", "Phred+33", "Phred+64"))

# Loop through Phred scores from 0 to 41
for (phred in 0:41) {
  # Calculate the probability of error
  prob_error <- 10^(phred / -10)

  # Convert Phred scores to ASCII characters
  ascii_phred33 <- intToUtf8(phred + 33)
  ascii_phred64 <- intToUtf8(phred + 64)

  # Print the results in a formatted table
  cat(sprintf("%-5d\t\t%0.5f\t\t%-6s\t\t%-10s\n",
              phred, prob_error,
              ascii_phred33, ascii_phred64))
}
```


### Historical note: Why did Illumina use a +64 offset?

The **offset** determines how a numeric quality score is stored as a character; it is separate from the formula used to calculate that score.

1. **Early Solexa FASTQ used a different quality scale.** Its log-odds score was $Q_{\mathrm{Solexa}}=-10\log_{10}(P_{\mathrm{error}}/(1-P_{\mathrm{error}}))$, and reported scores could be as low as **−5**. Standard Phred instead uses $Q=-10\log_{10}(P_{\mathrm{error}})$.
2. **The +64 offset accommodated negative scores as printable characters.** For example, −5 + 64 = 59, the ASCII code for `;`. With +33, −5 would become ASCII 28, a nonprinting control character. The exact choice of 64 was a format convention; negative scores did not uniquely require that number.
3. **Illumina pipeline 1.3–1.7 changed the score formula but retained +64.** These versions used standard Phred scores encoded as **Phred+64**. Thus, Solexa+64 and Phred+64 share an offset but are not the same quality scale.
4. **CASAVA 1.8 switched to Phred+33**, matching the Sanger FASTQ convention used by modern Illumina FASTQ files.

**Example:** A standard Phred score of Q30 is stored as `?` with +33 or `^` with +64. Both represent an estimated error probability of **0.001 (0.1%)**. Changing the encoding does not change the underlying quality.

**References:** [Cock et al. (2010), FASTQ format variants](https://doi.org/10.1093/nar/gkp1137); [Biopython QualityIO documentation and format history](https://raw.githubusercontent.com/biopython/biopython/master/Bio/SeqIO/QualityIO.py).